# MIMIC-IV Microbiology Exploration

This notebook performs EDA specifically on the `microbiologyevents` table to understand data sparsity, test frequencies, and the relationship between specimen types and tests. It also maps the specimens into 7 clinical panels for the P-CAFE Reinforcement Learning environment.

## Section 1: Setup & Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set plot style for better readability
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11

# Define base path to MIMIC-IV data
base_path = 'C:\\Users\\Eli\\Data\\physionet.org\\files\\mimiciv\\3.1\\hosp\\'

# Load microbiology events
print("Loading microbiologyevents.csv.gz ...")
micro_df = pd.read_csv(base_path + 'microbiologyevents.csv.gz')
print(f"Loaded microbiologyevents: {micro_df.shape[0]:,} rows, {micro_df.shape[1]} columns")

# Load admissions to get total unique hadm_ids
print("Loading admissions.csv.gz ...")
adm_df = pd.read_csv(base_path + 'admissions.csv.gz')
total_admissions = adm_df['hadm_id'].nunique()
print(f"Total unique hospital admissions (hadm_id): {total_admissions:,}")

# Filter out rows where hadm_id is null (events not linked to an admission)
micro_df = micro_df.dropna(subset=['hadm_id'])
print(f"After dropping null hadm_id: {micro_df.shape[0]:,} rows remain")
print("\nColumn overview:")
print(micro_df.dtypes)

## Section 2: Relation of `spec_type_desc` to `test_name`

In [ ]:
# --- Unique test_names per spec_type_desc ---
tests_per_spec = (
    micro_df.groupby('spec_type_desc')['test_name']
    .nunique()
    .sort_values(ascending=False)
)
print("Unique test_names per spec_type_desc:")
print(tests_per_spec.to_string())
print(f"\nTotal unique spec_type_desc values: {tests_per_spec.shape[0]}")

# --- Cross-tabulation: Top 15 spec_type_desc vs Top 15 test_name ---
top15_spec = micro_df['spec_type_desc'].value_counts().head(15).index.tolist()
top15_test = micro_df['test_name'].value_counts().head(15).index.tolist()

cross_tab = (
    micro_df[
        micro_df['spec_type_desc'].isin(top15_spec) &
        micro_df['test_name'].isin(top15_test)
    ]
    .pivot_table(
        index='spec_type_desc',
        columns='test_name',
        aggfunc='size',
        fill_value=0
    )
)

# Plot heatmap
fig, ax = plt.subplots(figsize=(18, 9))
sns.heatmap(
    cross_tab,
    annot=True,
    fmt='d',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Top 15 Specimen Types vs Top 15 Test Names (Event Counts)', fontsize=14, pad=15)
ax.set_xlabel('Test Name', fontsize=12)
ax.set_ylabel('Specimen Type', fontsize=12)
ax.tick_params(axis='x', labelrotation=45, labelsize=9)
ax.tick_params(axis='y', labelrotation=0, labelsize=9)
plt.tight_layout()
plt.show()

## Section 3: Appearances per Test (The Long Tail)

In [ ]:
# --- Test frequency distribution ---
test_counts = (
    micro_df.groupby('test_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

print(f"Total unique test_name values: {test_counts.shape[0]}")
print("\nSummary statistics of test frequencies:")
print(test_counts['count'].describe().to_string())
print(f"\nTop 10 most frequent tests:")
print(test_counts.head(10).to_string(index=False))
print(f"\nBottom 10 least frequent tests:")
print(test_counts.tail(10).to_string(index=False))

# --- Line chart: Long Tail of test frequencies ---
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(
    range(len(test_counts)),
    test_counts['count'].values,
    color='steelblue',
    linewidth=1.5
)
ax.set_yscale('log')
ax.set_title('Test Frequency Distribution — The Long Tail (Log Scale)', fontsize=14)
ax.set_xlabel('Test Rank (sorted by frequency, most frequent first)', fontsize=12)
ax.set_ylabel('Number of Events (log scale)', fontsize=12)
ax.axhline(y=100, color='red', linestyle='--', linewidth=1, label='100 events threshold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

rare_tests = (test_counts['count'] < 100).sum()
print(f"\nTests with fewer than 100 events: {rare_tests} ({rare_tests / len(test_counts) * 100:.1f}% of all tests)")

## Section 4: Data Sparsity & Clinical Yield

In [ ]:
total_events = len(micro_df)

# --- Yield Sparsity (Positivity Rate) ---
# Positive = org_name is not null AND does not contain 'NO GROWTH', 'CANCELLED', or 'VOID'
negative_keywords = ['NO GROWTH', 'CANCELLED', 'VOID']
has_org = micro_df['org_name'].notna()
is_negative_keyword = micro_df['org_name'].str.upper().str.contains(
    '|'.join(negative_keywords), na=False
)
positive_mask = has_org & ~is_negative_keyword
n_positive = positive_mask.sum()
yield_rate = n_positive / total_events * 100
print(f"=== Yield Sparsity (Positivity Rate) ===")
print(f"  Total microbiology events     : {total_events:,}")
print(f"  Events with organism isolated : {n_positive:,}")
print(f"  Positivity Rate               : {yield_rate:.2f}%")
print(f"  Yield Sparsity (negative rate): {100 - yield_rate:.2f}%")

# --- Susceptibility Sparsity ---
n_with_susceptibility = micro_df['ab_name'].notna().sum()
susceptibility_rate = n_with_susceptibility / total_events * 100
print(f"\n=== Susceptibility Sparsity ===")
print(f"  Events with antibiotic susceptibility data: {n_with_susceptibility:,}")
print(f"  Susceptibility Coverage Rate              : {susceptibility_rate:.2f}%")
print(f"  Susceptibility Sparsity                   : {100 - susceptibility_rate:.2f}%")

# --- Admission-level Sparsity ---
admissions_with_micro = micro_df['hadm_id'].nunique()
admission_coverage = admissions_with_micro / total_admissions * 100
print(f"\n=== Admission-level Sparsity ===")
print(f"  Total hospital admissions                  : {total_admissions:,}")
print(f"  Admissions with ≥1 microbiology event      : {admissions_with_micro:,}")
print(f"  Micro Coverage (% admissions with micro)   : {admission_coverage:.2f}%")
print(f"  Admission-level Sparsity                   : {100 - admission_coverage:.2f}%")

## Section 5: The 7 Clinical Panels (RL Action Space)

In [ ]:
# --- Define mapping from spec_type_desc to 7 canonical RL panels ---
panel_mapping = {
    # Blood
    'BLOOD CULTURE': 'blood',
    'BLOOD CULTURE - SEROLOGY': 'blood',
    'BLOOD CULTURE ( MYCO/F LYTIC BOTTLE)': 'blood',
    'BLOOD CULTURE (POST-MORTEM)': 'blood',
    'BLOOD CULTURE (NEUROLOGICAL)': 'blood',
    'FLUID RECEIVED IN BLOOD CULTURE BOTTLES': 'blood',
    'SEROLOGY': 'blood',

    # Urine
    'URINE': 'urine',
    'URINE,KIDNEY': 'urine',
    'URINE,SUPRAPUBIC ASPIRATE': 'urine',
    'URINE,VOIDED': 'urine',
    'URINE,CYSTOSCOPY': 'urine',
    'URINE,FOLEY': 'urine',
    'URINE (ILEAL CONDUIT)': 'urine',

    # Stool
    'STOOL': 'stool',
    'FECES': 'stool',
    'RECTAL SWAB': 'stool',

    # Respiratory
    'BRONCHOALVEOLAR LAVAGE': 'respiratory',
    'BRONCHIAL WASHINGS': 'respiratory',
    'BRONCHIAL BRUSH': 'respiratory',
    'BRONCHIAL BRUSH - PROTECTED': 'respiratory',
    'SPUTUM': 'respiratory',
    'THROAT': 'respiratory',
    'THROAT CULTURE': 'respiratory',
    'THROAT FOR STREP': 'respiratory',
    'PLEURAL FLUID': 'respiratory',
    'NASOPHARYNGEAL SWAB': 'respiratory',
    'NASAL SWAB': 'respiratory',
    'Mini-BAL': 'respiratory',
    'TRANSTRACHEAL ASPIRATE': 'respiratory',
    'TRACHEAL ASPIRATE': 'respiratory',
    'BRONCHIAL ASPIRATE': 'respiratory',
    'ENDOTRACHEAL': 'respiratory',
    'ENDOTRACHEAL TUBE': 'respiratory',
    'SINUS ASPIRATE': 'respiratory',

    # Swab & Screen
    'WOUND CULTURE': 'swab_and_screen',
    'SWAB': 'swab_and_screen',
    'ABSCESS': 'swab_and_screen',
    'TISSUE': 'swab_and_screen',
    'SKIN SCRAPINGS': 'swab_and_screen',
    'EYE': 'swab_and_screen',
    'PERITONEAL FLUID': 'swab_and_screen',
    'BILE': 'swab_and_screen',
    'JOINT FLUID': 'swab_and_screen',
    'BONE MARROW': 'swab_and_screen',
    'CATHETER TIP-IV': 'swab_and_screen',
    'FOREIGN BODY': 'swab_and_screen',
    'NAIL CLIPPINGS': 'swab_and_screen',
    'HAIR': 'swab_and_screen',
    'PACEMAKER': 'swab_and_screen',

    # CSF
    'CEREBROSPINAL FLUID (CSF)': 'csf',
    'CSF;SPINAL FLUID': 'csf',
}

# Apply mapping; anything not explicitly listed goes to 'other'
micro_df['micro_panel'] = micro_df['spec_type_desc'].map(panel_mapping).fillna('other')

print("Panel assignment distribution:")
print(micro_df['micro_panel'].value_counts().to_string())

# Identify spec_type_desc values that were mapped to 'other' for transparency
other_specs = (
    micro_df[micro_df['micro_panel'] == 'other']['spec_type_desc']
    .value_counts()
)
print(f"\nSpec types mapped to 'other' panel ({len(other_specs)} unique values):")
print(other_specs.to_string())

# --- Bar chart: Total events per panel ---
panel_order = ['blood', 'urine', 'respiratory', 'stool', 'swab_and_screen', 'csf', 'other']
panel_counts = micro_df['micro_panel'].value_counts().reindex(panel_order, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette('Set2', n_colors=len(panel_order))
bars = ax.bar(panel_counts.index, panel_counts.values, color=colors, edgecolor='black', linewidth=0.7)

# Annotate bars with counts
for bar, count in zip(bars, panel_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(panel_counts.values) * 0.01,
        f'{count:,}',
        ha='center', va='bottom', fontsize=10
    )

ax.set_title('Total Microbiology Events per Clinical Panel (7-Panel RL Action Space)', fontsize=14)
ax.set_xlabel('Clinical Panel', fontsize=12)
ax.set_ylabel('Number of Events', fontsize=12)
ax.tick_params(axis='x', labelrotation=15)
plt.tight_layout()
plt.show()

# --- Panel Sparsity: % of all admissions with ≥1 test from each panel ---
print("\n=== Panel Sparsity (% of all hospital admissions with ≥1 test per panel) ===")
print(f"(Based on {total_admissions:,} total unique hospital admissions)\n")

for panel in panel_order:
    panel_hadm = micro_df[micro_df['micro_panel'] == panel]['hadm_id'].nunique()
    coverage = panel_hadm / total_admissions * 100
    sparsity = 100 - coverage
    print(
        f"  {panel:<20s}: {panel_hadm:>6,} admissions  "
        f"({coverage:5.2f}% coverage  |  {sparsity:5.2f}% sparsity)"
    )